# Fine-tune the case-note student (QLoRA) - AML pipeline V2

**Not run yet.** Written to be run top to bottom on a Colab **L4** GPU (Runtime > Change runtime type).

What it does: trains a LoRA adapter on `train.jsonl` (chat format: system / user / assistant), evaluates on `validation.jsonl` after each epoch, saves the adapter to Google Drive, then writes the student's notes for the 50 dev alerts and, once, the 200 test alerts.

**Size and cost (estimate, not measured).** Train: 1,248 examples, 8.8M Qwen tokens per epoch, longest example 8,046 tokens (all fit 8,192). Expect about 9-10 h on an L4 for 2 epochs (about 550 tokens/s; range 7-11 h) and about 1 h to write the 250 notes: roughly 20 compute units (15-25). Run once with MAX_STEPS = 3 first to measure the real speed.

Put these in one Drive folder (`DRIVE_DIR` below), from `data/finetune/` of the repo: `train.jsonl`, `validation.jsonl`, `dev_inputs.jsonl`, `test_inputs.jsonl`, and `src/student_generate.py`.

**Model and licence.** Default `Qwen/Qwen2.5-7B-Instruct`: good at JSON, 128k context (we use 8,192). The cell below prints its licence from the Hugging Face model card; the 7B model is listed as Apache-2.0, but read the card before using the adapter beyond this project. Change `MODEL_NAME` to try another instruct model (the chat template and LoRA target names in the config must fit it).

**Never train on the test period.** `train.jsonl` and `validation.jsonl` come from HI-Small; the test inputs are the 200 LI-Small test alerts and are used only in the last cell.

In [ ]:
!pip -q install -U "transformers>=4.45" "peft>=0.13" "bitsandbytes>=0.43" accelerate huggingface_hub

In [ ]:
# ---- Settings (change here, nowhere else) ----
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # open instruct model, good JSON, long context
DRIVE_DIR = "/content/drive/MyDrive/aml_finetune"   # data in, adapter and notes out
MAX_SEQ_LEN = 8192          # prompt + note; longer examples are skipped, never cut
EPOCHS = 2
LEARNING_RATE = 2e-4        # usual for LoRA; the cosine schedule decays it
WARMUP_RATIO = 0.03
PER_DEVICE_BATCH = 1        # 8k tokens per example already fills the GPU
GRADIENT_ACCUMULATION = 16  # effective batch of 16 examples
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
SEED = 0
MAX_STEPS = -1            # -1 = full run; set 3 for a quick smoke test first
SAVE_EVERY_STEPS = 20       # about 1.2 h on an L4; a disconnect resumes from here
ADAPTER_DIR = DRIVE_DIR + "/adapter"          # final LoRA adapter
CHECKPOINT_DIR = DRIVE_DIR + "/checkpoints"   # lets a disconnect resume

In [ ]:
import json, os, torch
from google.colab import drive
from huggingface_hub import model_info

drive.mount("/content/drive")
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
info = model_info(MODEL_NAME)
print("Model:", MODEL_NAME, "| licence:", info.card_data.license if info.card_data else "unknown")
for name in ["train.jsonl", "validation.jsonl", "dev_inputs.jsonl", "test_inputs.jsonl", "student_generate.py"]:
    assert os.path.exists(os.path.join(DRIVE_DIR, name)), "missing in Drive: " + name

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset as TorchDataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def read_jsonl(name):
    with open(os.path.join(DRIVE_DIR, name)) as handle:
        return [json.loads(line) for line in handle]

def encode(example):
    # Loss only on the assistant's note: the prompt tokens get label -100.
    system_user, full = example["messages"][:2], example["messages"]
    prompt = tokenizer.apply_chat_template(system_user, tokenize=False, add_generation_prompt=True)
    text = tokenizer.apply_chat_template(full, tokenize=False).rstrip("\n")
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    assert ids[:len(prompt_ids)] == prompt_ids, "prompt is not a prefix of the full chat"
    return {"input_ids": ids, "labels": [-100] * len(prompt_ids) + ids[len(prompt_ids):]}

class Encoded(TorchDataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def build(name):
    rows = [encode(e) for e in read_jsonl(name)]
    kept = [r for r in rows if len(r["input_ids"]) <= MAX_SEQ_LEN]
    lengths = sorted(len(r["input_ids"]) for r in rows)
    print(f"{name}: {len(kept)} of {len(rows)} fit {MAX_SEQ_LEN} tokens "
          f"(p50 {lengths[len(lengths)//2]}, max {lengths[-1]})")
    return Encoded(kept)

train_data, validation_data = build("train.jsonl"), build("validation.jsonl")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quantization, device_map="auto",
    torch_dtype=torch.bfloat16, attn_implementation="sdpa",
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

arguments = TrainingArguments(
    output_dir=CHECKPOINT_DIR, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine", warmup_ratio=WARMUP_RATIO, seed=SEED, bf16=True,
    per_device_train_batch_size=PER_DEVICE_BATCH, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION, gradient_checkpointing=True,
    optim="paged_adamw_8bit", eval_strategy="epoch", save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS, max_steps=MAX_STEPS, save_total_limit=3, logging_steps=10, report_to="none",
)
trainer = Trainer(
    model=model, args=arguments, train_dataset=train_data, eval_dataset=validation_data,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
last = get_last_checkpoint(CHECKPOINT_DIR) if os.path.isdir(CHECKPOINT_DIR) else None
print("Resuming from", last) if last else print("Starting fresh")
trainer.train(resume_from_checkpoint=last)

In [ ]:
trainer.save_model(ADAPTER_DIR)          # the LoRA adapter only (about 150 MB)
tokenizer.save_pretrained(ADAPTER_DIR)
history = [h for h in trainer.state.log_history if "eval_loss" in h]
print("Validation loss by epoch:", [(h["epoch"], round(h["eval_loss"], 4)) for h in history])
print("Saved adapter to", ADAPTER_DIR)
# Rising validation loss in epoch 2 means overfitting: use the epoch-1 checkpoint in
# CHECKPOINT_DIR instead (point --adapter at it below).

## Student notes (inference)

Free the training model first (Runtime > Restart session is fine; the adapter is on Drive). The dev run picks nothing automatically: look at it before the test run. **The test run is meant to happen once.**

In [ ]:
# 50 dev alerts (LI-Small validation period)
!python {DRIVE_DIR}/student_generate.py --inputs {DRIVE_DIR}/dev_inputs.jsonl --output {DRIVE_DIR}/dev_raw.jsonl --adapter {ADAPTER_DIR} --base-model {MODEL_NAME}

In [ ]:
# 200 test alerts: run ONCE, after the dev result looks right. It skips finished alerts.
!python {DRIVE_DIR}/student_generate.py --inputs {DRIVE_DIR}/test_inputs.jsonl --output {DRIVE_DIR}/test_raw.jsonl --adapter {ADAPTER_DIR} --base-model {MODEL_NAME}

## Score on your own computer (CPU)

Download `dev_raw.jsonl` and `test_raw.jsonl` into `data/finetune/` of the repo, then:

```bash
python -m src.score_student --set dev  --raw data/finetune/dev_raw.jsonl
python -m src.score_student --set test --raw data/finetune/test_raw.jsonl
```

Same fact check and the same five measures (`src/eval_v2.py`) as V1 and prompt v3.